# BlueCarbon-AI: train the v2 model on a free GPU

This notebook runs the whole pipeline on Google Colab:

1. It downloads Sentinel-2 composites and fused reference labels for 11 coastal sites with Google Earth Engine.
2. It cuts them into chips with a leakage-free spatial split. Moreton Bay and Tampa Bay are fully held out.
3. It trains a U-Net (ResNet-34) and reports per-class IoU/F1 on the held-out data.
4. It runs the Mission Bay 2018→2024 change case study and exports demo bundles for the web app.

**Before you start:** go to *Runtime → Change runtime type → T4 GPU*. You also need an Earth Engine-enabled Google Cloud project; the default is `bluecarbon-ai`.

Expect roughly 20–40 minutes for downloads and 30–60 minutes for training on a T4.

In [ ]:
!git clone -q https://github.com/yanicksanchez14-creator/bluecarbon-ai.git
%cd bluecarbon-ai
!pip -q install -e ".[gee]"


In [ ]:
import ee
PROJECT = "bluecarbon-ai"   # <- your Earth Engine cloud project
ee.Authenticate()
ee.Initialize(project=PROJECT)
print("Earth Engine OK:", ee.Number(1).getInfo() == 1)


## 1 · Download imagery + reference labels

In [ ]:
!bluecarbon fetch -c configs/default.yaml --sites configs/sites.yaml

Take a quick look at one site: imagery on the left, fused labels on the right.

In [ ]:
import rasterio, matplotlib.pyplot as plt
from bluecarbon.viz import true_color, class_rgba
site = "ten_thousand_isl_fl"
with rasterio.open(f"runs/default/sites/{site}/image.tif") as s: img = s.read()
with rasterio.open(f"runs/default/sites/{site}/label.tif") as s: lab = s.read(1)
fig, ax = plt.subplots(1, 2, figsize=(14, 7))
ax[0].imshow(true_color(img)); ax[1].imshow(true_color(img)); ax[1].imshow(class_rgba(lab, 190))
for a in ax: a.axis("off")


## 2 · Chips + train

In [ ]:
!bluecarbon chips -c configs/default.yaml --sites configs/sites.yaml
!bluecarbon train -c configs/default.yaml

In [ ]:
import json, numpy as np
from bluecarbon.viz import plot_confusion
from bluecarbon.schema import CLASS_KEYS
m = json.load(open("runs/default/model/metrics.json"))
print("TEST (held-out blocks + held-out sites)")
print({k: m["test"][k] for k in ("mIoU", "macro_f1", "overall_accuracy", "kappa")})
print("IoU:", m["test"]["iou"])
for site, s in m["test_per_site"].items(): print(site, "mIoU", s["mIoU"])
plot_confusion(np.array(m["test_confusion"]), CLASS_KEYS, "docs/img/confusion_test.png")
from IPython.display import Image; Image("docs/img/confusion_test.png")


## 3 · Case study + demo bundles

In [ ]:
!bluecarbon case-study -m runs/default/model/model.pt
!echo '{"t0": "Summer 2018", "t1": "Summer 2024"}' > runs/default/scenes/mission_bay_change/periods.json
!bluecarbon export-demo runs/default/scenes/mission_bay_change --name mission_bay_change \
    --title "Mission Bay, San Diego (2018 → 2024)" -m runs/default/model/model.pt \
    --description "Salt marsh and seagrass change in Mission Bay between summer 2018 and summer 2024."

# One mapped scene per held-out test site, so the demo shows generalization
for name, bbox, title in [("moreton_bay_au", "153.1 -27.5 153.35 -27.25", "Moreton Bay, Australia (held-out)"),
                          ("tampa_bay_fl", "-82.7 27.6 -82.5 27.8", "Tampa Bay, Florida (held-out)")]:
    b = " ".join(f"--bbox {v}" for v in bbox.split())
    !bluecarbon scene {b} --start 2024-01-01 --end 2025-01-01 -m runs/default/model/model.pt --name {name}
    !bluecarbon export-demo runs/default/scenes/{name} --name {name} --title "{title}" -m runs/default/model/model.pt \
        --description "Site never seen during training. Sentinel-2 2024 composite."


## 4 · Publish

Run the cell below to download `model.pt` and `demo_data.zip`. Then:
1. On GitHub: **Releases → Draft a new release**, tag it `v2.0.0`, attach `model.pt`, and publish it.
2. Unzip `demo_data.zip` into the repo's `demo_data/` folder and commit it. Or just ask Claude to do it.
3. In the Streamlit Cloud app settings, set the secret `MODEL_URL` to the release asset URL.

In [ ]:
!cp runs/default/model/model.pt model.pt && cp runs/default/model/metrics.json metrics.json
!zip -qr demo_data.zip demo_data docs/img
from google.colab import files
for f in ["model.pt", "metrics.json", "demo_data.zip"]: files.download(f)
